In [0]:
import pyspark.sql.functions as F
import json

In [0]:
# Read Kafka connection details from Databricks secrets instead of hardcoding credentials.
Kafka_connection_json = dbutils.secrets.get(scope = "finguard-scope", key = "kafka_connection_details")
kafka_config = json.loads(Kafka_connection_json)

bootstrap_servers = kafka_config["bootstrap_servers"]
api_key = kafka_config["api_key"]
api_secret = kafka_config["api_secret"]
topic_name = kafka_config["topic"]

jass_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
# Read a batch of credit card transaction messages from the Confluent Cloud Kafka topic.
# Connects using SASL_SSL with PLAIN authentication, loads all available records into a DataFrame.
sample_batch_df = (spark.read.format('kafka')
                   .option("kafka.bootstrap.servers", bootstrap_servers)
                   .option("subscribe", topic_name)
                   .option("kafka.security.protocol", "SASL_SSL")
                   .option("kafka.sasl.mechanism", "PLAIN")
                   .option("kafka.sasl.jaas.config", jass_config)
                   .load())

display(sample_batch_df)

In [0]:
print(sample_batch_df.count())
sample_batch_df.printSchema()


In [0]:
parsed_df = sample_batch_df.select(
    F.col("key").cast("string"),
    F.col("value").cast("string"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp")
)
display(parsed_df)
print(parsed_df.select('key').distinct().count())

In [0]:
# %sql
# CREATE CATALOG IF NOT EXISTS finuard_project;
# USE CATALOG finuard_project;
# CREATE SCHEMA IF NOT EXISTS test;

In [0]:
# Write the parsed Kafka transaction data to a Delta table in Unity Catalog.
# Overwrites any existing data in the target table on each run.
parsed_df.write.mode("overwrite").saveAsTable("finuard_project.test.transactions_batch")

In [0]:
display(spark.sql("select * from finuard_project.test.transactions_batch"))
display(spark.sql("describe history finuard_project.test.transactions_batch"))

In [0]:
# Define a streaming read from the Confluent Cloud Kafka topic.
# Uses SASL_SSL with PLAIN authentication (same credentials as the batch read)
# and starts from the earliest available offsets so no historical messages are missed.
sample_stream_df = (spark.readStream.format('kafka')
                   .option("kafka.bootstrap.servers", bootstrap_servers)
                   .option("subscribe", topic_name)
                   .option("kafka.security.protocol", "SASL_SSL")
                   .option("kafka.sasl.mechanism", "PLAIN")
                   .option("kafka.sasl.jaas.config", jass_config)
                   .option("startingOffsets", "earliest")
                   .load())

In [0]:
parsed_stream_df = sample_stream_df.select(
    F.col("key").cast("string"),
    F.col("value").cast("string"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp")
)

In [0]:
# Write the streaming Kafka transaction data to a Delta table in Unity Catalog.
# - outputMode("Append"): only new rows are added to the target table on each micro-batch.
# - checkPointLocation: stores streaming progress/checkpoints so the query can recover after failures or restarts without reprocessing already-committed data.
# - trigger(availableNow=True): processes all available data in a single micro-batch and then stops, useful for a one-time backfill or scheduled run.
# - toTable: creates or appends to the specified Unity Catalog table.
(
    parsed_stream_df.writeStream.outputMode("Append")
    .option(
        "checkPointLocation",
        "/Volumes/finuard_project/test/transactions_vol/checkpoint_loc/",
    )
    .trigger(availableNow= True )
    .toTable("finuard_project.test.transactions_stream")
)

In [0]:
display(spark.sql("select * from finuard_project.test.transactions_stream"))
display(spark.sql("describe history finuard_project.test.transactions_stream"))